<a href="https://colab.research.google.com/github/ravindravala/AI/blob/main/CIFAR_100.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader,Dataset
import torchvision
import torchvision.transforms as transforms

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
device

device(type='cuda')

In [4]:
transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
])

In [5]:
train_dataset = torchvision.datasets.CIFAR100(root="./data", download=True,train=True,transform=transform)
test_dataset = torchvision.datasets.CIFAR100(root="./data", download=True,train=False,transform=transform)


In [6]:
train_loader = DataLoader(train_dataset,batch_size=64,shuffle=True)
test_loader = DataLoader(test_dataset,batch_size=64,shuffle=False)

In [7]:
img,lbl = next(iter(train_loader))
img.shape,lbl.shape

(torch.Size([64, 3, 32, 32]), torch.Size([64]))

In [8]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class BetterCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv_block1 = nn.Sequential(
            nn.Conv2d(3, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2)   # 32 → 16
        )

        self.conv_block2 = nn.Sequential(
            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2)   # 16 → 8
        )

        self.conv_block3 = nn.Sequential(
            nn.Conv2d(128, 256, 3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2)   # 8 → 4
        )

        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 4 * 4, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, 100)
        )

    def forward(self, x):
        x = self.conv_block1(x)
        x = self.conv_block2(x)
        x = self.conv_block3(x)
        x = self.fc(x)
        return x

In [9]:
model = BetterCNN().to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()


In [10]:
epochs=40

for epoch in range(epochs):
  total=0
  for x,y in train_loader:
    x,y = x.to(device),y.to(device)
    optimizer.zero_grad()
    pred = model(x)
    loss = criterion(pred,y)
    total += loss.item()
    loss.backward()
    optimizer.step()

  if epoch % 5 == 0:
    print(f"Epoch: {epoch+1}, Loss: {total/len(train_loader)}")

Epoch: 1, Loss: 4.113796515842838
Epoch: 6, Loss: 3.234418394315578
Epoch: 11, Loss: 2.8737112893472854
Epoch: 16, Loss: 2.606345187977452
Epoch: 21, Loss: 2.3766001098601106
Epoch: 26, Loss: 2.191552254702429
Epoch: 31, Loss: 2.0195042832428234
Epoch: 36, Loss: 1.8834723142711707


In [11]:
model.eval()
with torch.no_grad():
  correct = 0
  total =0

  for x,y in test_loader:
    x,y = x.to(device),y.to(device)
    pred = model(x)
    _,predictions = torch.max(pred,1)
    correct += (predictions == y).sum().item()
    total += y.shape[0]

  print(f"Efficiency: {100 * (correct/total)}")

Efficiency: 52.87
